# Module 13: Final Research Report and Project Package

目标：将已经运行的研究整理为可以阅读、展示和复核的最终报告。这个模块是本版本最后一个模块，不增加新的回归或选择更显著的结果。
沿用 8 个 Code Cells；中文逐格讲解，英文代码、批注、图表、报告和文件名。主要使用 pandas、普通循环和字符串，不新增分析函数或安装新包。

最终交付会包含英文 Markdown 报告、可独立打开的 HTML 报告、三张结果图、来源文件哈希清单和复现说明。
HTML 内嵌图片，可复制单个 HTML 文件给导师；Markdown 报告展示图片时，需要与 figures 子目录一起保存。

**当前版本的完成边界：** 分析流程和报告可以完成，整体人工审核仍未完成。
现有结果为毛利率 401 对／36 家、现金流 490 对／42 家，共同样本 399 对；已审核 2 家公司、7 个年度、3 条案例。
保留 provisional 样本和尚未审核记录是本版本的明确限制。报告不会把未审核的 AKA、ARHS、ULTA、KSS 编写成已确认经营机制。
两个充分控制模型的 95% 区间都包含零，结论是证据不足以支持稳定关联；不能据此证明没有关系、预测有效或因果成立。

**运行顺序：** Module12 → Module10 → Module11 → Module13。
若人工审核文件有新修改，先让 Module12 应用决定，再重新跑 Module10 和 11，最后重新运行本模块全部 8 格。
仅运行 Module13 不会补值、审核年报或重新拟合模型。
完整 notebook 为当前项目调整了运行顺序：1–9 → 12 → 10 → 11 → 13。
它依赖已经建立的原始缓存、审核工作表和初始 Module11 案例文件；首次从空目录复现需要先完成原 Guides 的 1–11，再建立 Module12。


## What the Final Report Will Say

报告以研究问题、样本与方法、描述结果、控制后的结果、敏感性、已核查案例、限制和实际价值为主线。
不会将当前 R² 写成预测准确率；不会因案例方向符合或不符合假设就决定排除。
“Final report”表示这一版报告交付完成；“Partial filing review”描述证据审核范围，二者会同时出现在报告中。
这是一份公司基本面研究报告，没有股票收益回测、买卖建议或样本外预测验证。


## Cell 1: Set the Input and Report Paths

本项目实际根目录为 [project root]，这里明确写出，避免在 Downloads 或 Guide 目录运行时读错。
若迁移到其他电脑，只改 PROJECT_ROOT。REPORT_DATE 是报告封面日期；之后更新版本时也修改这一行。
DATA_DIR 读取 Module12 重建并经过 Module10、11 分析的结果；REPORT_DIR 保存最终交付，figures 保存本模块的图。
mkdir 创建缺少的目录，不清空目录。重新运行会更新本模块自己的报告和图。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import base64
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
REPORT_DATE = "2026-10-11"
DATA_DIR = PROJECT_ROOT / "work" / "data" / "processed" / "module12_reviewed"
REVIEW_DIR = DATA_DIR.parent / "module12_manual_review"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Report inputs:", DATA_DIR)
print("Report outputs:", REPORT_DIR)


## Cell 2: Read the Applied Results and Confirm Their Versions

tables 将短名称对应到结果文件，一次读取所需 DataFrame；代码中的文件列表也会用于最后的来源清单。
keep_default_na=False 让空备注保持空字符串；含金额的列稍后仍以数值计算。
metadata12 负责人工审核状态，metadata10 和 metadata11 记录分析输入。检查嵌套版本一致，可识别“改了审核但忘了重跑分析”。
版本检查失败时按 12→10→11 重跑，不要删除 assert 来继续出报告。
核对四个审核表的正式决定与应用结果，避免人工表新保存后报告仍显示旧审核。
检查只针对会影响报告正确性的状态和结果，没有要求完成所有 pending 项目。


In [ ]:
input_files = {
    "annual": "expanded_module12_annual_audit.csv",
    "gm_pairs": "expanded_gm_next_year_pairs.csv",
    "ocf_pairs": "expanded_ocf_next_year_pairs.csv",
    "readiness": "expanded_module12_report_readiness.csv",
    "pooled": "expanded_inventory_group_summary.csv",
    "balanced": "expanded_company_balanced_summary.csv",
    "models": "expanded_baseline_regression_results.csv",
    "sensitivity": "expanded_adjusted_regression_sensitivity.csv",
    "cases": "expanded_module12_case_results.csv",
    "companies": "candidate_scope_review.csv",
    "applied": "expanded_module12_applied_values.csv",
}
tables = {}
source_paths = []
for name, filename in input_files.items():
    path = DATA_DIR / filename
    tables[name] = pd.read_csv(path, keep_default_na=False)
    source_paths.append(path)
metadata = {}
for module in [12, 10, 11]:
    path = DATA_DIR / f"expanded_module{module}_metadata.json"
    metadata[module] = json.loads(path.read_text(encoding="utf-8"))
    source_paths.append(path)
assert metadata[10]["module9_metadata"] == metadata[12], "Run Module 10 after the latest Module 12."
assert metadata[11]["module10_metadata"] == metadata[10], "Run Module 11 after the latest Module 10."
for filename, table_name, keys, fields in [
    ("company_review.csv", "companies", ["ticker"], ["eligibility_review_status", "include_decision", "eligibility_evidence_url", "eligibility_notes"]),
    ("annual_scope_review.csv", "annual", ["ticker", "period_end"], ["scope_status", "scope_evidence_url", "scope_notes"]),
    ("case_review.csv", "cases", ["ticker", "period_end"], ["review_status", "suggested_action", "units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "current_evidence_url", "next_evidence_url", "review_notes"]),
]:
    path = REVIEW_DIR / filename
    manual = pd.read_csv(path, keep_default_na=False)
    left = manual[keys + fields].sort_values(keys).reset_index(drop=True)
    right = tables[table_name][keys + fields].sort_values(keys).reset_index(drop=True)
    assert left.equals(right), "New manual decisions need Module 12, then Modules 10 and 11: " + filename
    source_paths.append(path)
metric_path = REVIEW_DIR / "metric_review.csv"
metric_review = pd.read_csv(metric_path, keep_default_na=False)
approved = metric_review.loc[metric_review["review_status"].eq("approved")]
assert len(approved) == len(tables["applied"]), "Apply new metric approvals in Module 12 before reporting."
source_paths.append(metric_path)
for name, outcome in [("gm_pairs", "Gross margin"), ("ocf_pairs", "OCF-to-revenue")]:
    counts = tables["models"].loc[tables["models"]["outcome"].eq(outcome), "pairs"]
    assert counts.eq(len(tables[name])).all(), "Model sample counts do not match the applied pairs."
print("Applied decisions and saved analysis versions agree.")


## Cell 3: Summarize the Sample and the Actual Review Coverage

对两个结果分别统计配对数、公司数、共同样本、期间范围、已审核配对和披露时点。
review_complete_pair 沿用 Module12 定义：公司资格与上一期／本期／下一期年度审核记录通过；不等于独立审计。
样本数量从当前输入计算，不把 401／490 写死，因此后续合理补值或排除后能自然更新。
review_summary 同时列出已审核、总量和仍待审核数量；成本与毛利 missing 来自完整年度表，不误写成分析样本中的全部结果都缺失。


In [ ]:
sample_rows = []
for name, outcome in [("gm_pairs", "Gross margin"), ("ocf_pairs", "OCF-to-revenue")]:
    table = tables[name]
    sample_rows.append({
        "Outcome": outcome, "Pairs": len(table), "Companies": table["ticker"].nunique(),
        "Reviewed pairs": int(table["review_complete_pair"].sum()),
        "Current end from": table["period_end"].min(), "Current end to": table["period_end"].max(),
        "Latest target end": table["next_period_end"].max(),
        "Before target start": int(table["available_before_target_start"].sum()),
        "Different lengths": int(table["pair_period_length_changed"].sum()),
    })
sample_summary = pd.DataFrame(sample_rows)
common_keys = tables["gm_pairs"][["ticker", "period_end"]].merge(
    tables["ocf_pairs"][["ticker", "period_end"]], on=["ticker", "period_end"], validate="one_to_one"
)
reviewed_cases = tables["cases"].loc[tables["cases"]["review_status"].eq("reviewed")].copy()
pending_cases = tables["cases"].loc[tables["cases"]["review_status"].eq("pending")].copy()
review_summary = pd.DataFrame([
    {"Review item": "Company eligibility", "Reviewed": int(tables["companies"]["eligibility_review_status"].eq("reviewed").sum()), "Total": len(tables["companies"])},
    {"Review item": "Annual scope", "Reviewed": int(tables["annual"]["scope_status"].eq("reviewed").sum()), "Total": len(tables["annual"])},
    {"Review item": "Selected cases", "Reviewed": len(reviewed_cases), "Total": len(tables["cases"])},
])
review_summary["Pending"] = [
    int(tables["companies"]["eligibility_review_status"].eq("pending").sum()),
    int(tables["annual"]["scope_status"].eq("pending").sum()), len(pending_cases),
]
missing_cost = int(pd.to_numeric(tables["annual"]["cost_of_sales"], errors="coerce").isna().sum())
missing_gp = int(pd.to_numeric(tables["annual"]["gross_profit"], errors="coerce").isna().sum())
display(sample_summary)
display(review_summary)
print("Common outcome pairs:", len(common_keys))
print("Remaining missing cost / gross profit:", missing_cost, "/", missing_gp)


## Cell 4: Prepare the Main Results and Two Presentation Figures

main_results 只取充分控制的 Next level: company/year 模型，保留每增加 10pp 存货增长差的估计关联和公司聚类 95% 区间。
这是控制基期后的下一年比率模型；不能把数字写成收益率、概率变化或因果效应。
第一张图保留不同控制规格，展示结论如何随控制条件改变；现金流仅控制基期时仍是正向，不能说“加入基期就消失”。
第二张图是简单分组平均变化，只作描述；它和控制后回归回答的问题不同。
报告里的表和图都显示百分比或百分点单位。R² 作为样本内拟合指标，不当作预测准确率。


In [ ]:
main_results = tables["models"].loc[tables["models"]["model"].eq("Next level: company/year")].copy()
assert len(main_results) == 2, "Expected one adjusted model for each outcome."
main_table = main_results[["outcome", "pairs", "companies", "association_10pp", "ci_lower_10pp", "ci_upper_10pp"]].rename(columns={
    "outcome": "Outcome", "pairs": "Pairs", "companies": "Companies", "association_10pp": "Association per 10pp (pp)",
    "ci_lower_10pp": "95% lower (pp)", "ci_upper_10pp": "95% upper (pp)",
})
pooled_table = tables["pooled"][["outcome", "inventory_group", "pairs", "mean_change_pp", "median_change_pp", "decline_rate"]].copy()
pooled_table["decline_rate"] *= 100
pooled_table.columns = ["Outcome", "Inventory group", "Pairs", "Mean change (pp)", "Median change (pp)", "Decline rate (%)"]
balanced_table = tables["balanced"].rename(columns={
    "outcome": "Outcome", "companies": "Eligible companies", "mean_company_difference_pp": "Mean company difference (pp)",
    "median_company_difference_pp": "Median company difference (pp)", "share_negative": "Share negative (%)",
}).copy()
balanced_table["Share negative (%)"] *= 100
caption = f"Source: Modules 10-12; current ends {sample_summary['Current end from'].min()} to {sample_summary['Current end to'].max()}; partial filing review."
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, outcome in zip(axes, ["Gross margin", "OCF-to-revenue"]):
    chart = tables["models"].loc[tables["models"]["outcome"].eq(outcome) & tables["models"]["model"].str.startswith("Next level")]
    error = [chart["association_10pp"] - chart["ci_lower_10pp"], chart["ci_upper_10pp"] - chart["association_10pp"]]
    ax.errorbar(chart["association_10pp"], range(len(chart)), xerr=error, fmt="o", color="#245a73", capsize=4)
    ax.set_yticks(range(len(chart)), chart["model"].str.replace("Next level: ", "", regex=False))
    ax.set_xlabel("Estimated association per 10pp inventory gap (pp)")
    ax.set_ylabel("Model specification")
    ax.set_title(outcome)
    ax.axvline(0, color="gray", linewidth=1)
fig.suptitle("Next-year ratios: estimates and firm-clustered 95% intervals")
fig.text(0.5, 0.015, caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 0.92])
fig.savefig(FIGURE_DIR / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, outcome in zip(axes, ["Gross margin", "OCF-to-revenue"]):
    chart = tables["pooled"].loc[tables["pooled"]["outcome"].eq(outcome)]
    ax.bar(chart["inventory_group"], chart["mean_change_pp"], color=["#a9b2b8", "#245a73"])
    ax.set_xlabel("Current inventory growth minus revenue growth")
    ax.set_ylabel("Mean next-year ratio change (pp)")
    ax.set_title(outcome)
    ax.axhline(0, color="gray", linewidth=1)
fig.suptitle("Descriptive group means: annual pairs receive equal weight")
fig.text(0.5, 0.015, caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 0.92])
fig.savefig(FIGURE_DIR / "group_means.png", dpi=160, bbox_inches="tight")
plt.show()
display(main_table.round(4))


## Cell 5: Summarize Sensitivity Without Selecting a Preferred Result

完整显示四种预先安排的样本条件及各自数量，不按区间是否包含零决定保留结果。
zero_in_interval 表示区间包含零，不代表效果严格等于零，也不是假设成立的概率。
按结果统计区间包含零的行数，结论随实际输入生成，不写死“八条全部包含零”。
Unchanged period lengths 样本可能仍有 365／366 天差异；原规则标记的是超过三天的明显年度长度变化。
共同样本帮助检查两个结果的样本构成差异；中间 98% 使用信号的百分位，不按未来表现去掉极端值。


In [ ]:
sensitivity = tables["sensitivity"].copy()
sensitivity["zero_in_interval"] = sensitivity["ci_lower_10pp"].le(0) & sensitivity["ci_upper_10pp"].ge(0)
sensitivity_table = sensitivity[["outcome", "sample", "pairs", "companies", "association_10pp", "ci_lower_10pp", "ci_upper_10pp"]].copy()
sensitivity_table.columns = ["Outcome", "Sample rule", "Pairs", "Companies", "Association per 10pp (pp)", "95% lower (pp)", "95% upper (pp)"]
sensitivity_notes = []
for outcome, group in sensitivity.groupby("outcome", sort=False):
    count = int(group["zero_in_interval"].sum())
    sensitivity_notes.append(f"{outcome}: {count} of {len(group)} adjusted sensitivity intervals include zero.")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, outcome in zip(axes, ["Gross margin", "OCF-to-revenue"]):
    chart = sensitivity.loc[sensitivity["outcome"].eq(outcome)]
    labels = chart["sample"] + " (n=" + chart["pairs"].astype(str) + ")"
    error = [chart["association_10pp"] - chart["ci_lower_10pp"], chart["ci_upper_10pp"] - chart["association_10pp"]]
    ax.errorbar(chart["association_10pp"], range(len(chart)), xerr=error, fmt="o", color="#245a73", capsize=4)
    ax.set_yticks(range(len(chart)), labels)
    ax.set_xlabel("Adjusted association per 10pp inventory gap (pp)")
    ax.set_ylabel("Sample rule and pair count")
    ax.set_title(outcome)
    ax.axvline(0, color="gray", linewidth=1)
fig.suptitle("Adjusted sensitivity estimates and firm-clustered 95% intervals")
fig.text(0.5, 0.015, caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 0.92])
fig.savefig(FIGURE_DIR / "sensitivity.png", dpi=160, bbox_inches="tight")
plt.show()
display(sensitivity_table.round(4))
print(" ".join(sensitivity_notes))


## Cell 6: Separate Reviewed Cases from Pending Case Work

只为 review_status=reviewed 的案例展示有来源的经营解释。三年范围核查进度由配对表匹配，不能仅凭案例状态宣称完整核查。
次年变化在 CSV 中是小数，展示时乘 100 转成百分点；inventory_gap_pp 已是百分点，不再乘 100。
pending 表仅显示公司、日期与选择依据，作为剩余工作清单，不推断经营原因。
URBN 和 SPWH 是目的性选择的案例，不代表整体行业的发生率。报告保留他们的管理层解释、报表事实和原有局限说明。


In [ ]:
reviewed_keys = pd.concat([
    tables["gm_pairs"][["ticker", "period_end", "review_complete_pair"]],
    tables["ocf_pairs"][["ticker", "period_end", "review_complete_pair"]],
]).drop_duplicates(["ticker", "period_end"])
reviewed_cases = reviewed_cases.merge(reviewed_keys, on=["ticker", "period_end"], how="left", validate="one_to_one")
case_table = reviewed_cases[["ticker", "period_end", "inventory_gap_pp", "next_gross_margin_change", "next_ocf_margin_change", "suggested_action", "review_complete_pair"]].copy()
for column in ["next_gross_margin_change", "next_ocf_margin_change"]:
    case_table[column] = pd.to_numeric(case_table[column], errors="coerce") * 100
case_table.columns = ["Ticker", "Current period end", "Inventory gap (pp)", "Next GM change (pp)", "Next OCF/revenue change (pp)", "Decision", "Three-year review passed"]
pending_table = pending_cases[["ticker", "period_end", "selection_reason"]].copy()
pending_table.columns = ["Ticker", "Current period end", "Selection reason"]
display(case_table.round(4))
display(pending_table)


## Cell 7: Write the English Report in Markdown and HTML

本格较长主要因为包含英文报告正文；分析仍沿用前面的 DataFrame，不新增模型。
report_sections 保存章节名与英文段落；f-string 把当前样本量、系数、审核数量放入正文。
report_tables 把结果表放到相应章节。Markdown 表用普通循环拼接；HTML 使用 pandas.to_html，避免依赖额外包。
escape 转义 HTML 特殊字符，让备注中的 &、<、> 等显示为文字。base64 把三张图片放进 HTML，便于单文件分享。
报告没有自动编写 pending 案例。已审核案例直接引用已保存的英文证据和原始年报链接，管理层解释与因果结论保持区分。
sources 记录本次实际输入路径与哈希，hashlib.sha256 用于确认文件版本，不证明财务内容正确。


In [ ]:
adjusted_sentences = []
for _, row in main_results.iterrows():
    conclusion = "The 95% interval includes zero; the estimate remains imprecise."
    if row["ci_lower_10pp"] > 0 or row["ci_upper_10pp"] < 0:
        conclusion = "The 95% interval excludes zero in this specification; robustness and design limits still matter."
    adjusted_sentences.append(
        f"{row['outcome']}: a 10 percentage-point inventory-growth gap is associated with {row['association_10pp']:+.4f} percentage points in the next-year ratio, conditional on the controls (95% interval {row['ci_lower_10pp']:+.4f} to {row['ci_upper_10pp']:+.4f}). {conclusion}"
    )
all_intervals_include_zero = bool(sensitivity["zero_in_interval"].all())
if all_intervals_include_zero:
    final_conclusion = "The adjusted specifications examined do not provide sufficiently precise evidence of a stable association. This does not establish that the true association is zero or that inventory information has no research value."
else:
    final_conclusion = "The specifications provide mixed evidence that must be assessed using the full set of estimates and intervals, rather than a selected significant result."
company_count = len(tables["companies"])
annual_count = len(tables["annual"])
company_review_count = int(review_summary.loc[review_summary["Review item"].eq("Company eligibility"), "Reviewed"].iloc[0])
annual_review_count = int(review_summary.loc[review_summary["Review item"].eq("Annual scope"), "Reviewed"].iloc[0])
pending_names = ", ".join(pending_cases["ticker"].drop_duplicates()) or "None"
timing_min = int(min(tables[name]["days_after_target_start"].min() for name in ["gm_pairs", "ocf_pairs"]))
timing_max = int(max(tables[name]["days_after_target_start"].max() for name in ["gm_pairs", "ocf_pairs"]))
report_sections = {
    "Executive Summary": f"This project studies inventory growth relative to revenue growth and subsequent operating performance among {company_count} manually selected U.S.-listed merchandise retailers and consumer brands. It uses {annual_count} annual records, separate gross-margin and OCF-to-revenue research samples, and {len(common_keys)} common outcome pairs.\n\n" + "\n\n".join(adjusted_sentences) + f"\n\nFiling review covers {company_review_count} company eligibility decisions, {annual_review_count} annual rows and {len(reviewed_cases)} of {len(tables['cases'])} selected cases. The statistical sample remains provisional. " + final_conclusion,
    "Research Question and Economic Rationale": "Does inventory growth exceeding revenue growth in year t relate to changes in gross margin and operating cash flow divided by revenue in year t+1?\n\nInventory-growth gap = (Inventory_t / Inventory_(t-1) - 1) - (Revenue_t / Revenue_(t-1) - 1). Growth differences are expressed in percentage points. Gross margin = reported gross profit / analysis revenue. OCF-to-revenue = operating cash flow / analysis revenue. Change outcomes equal the next-year ratio minus the current-year ratio.\n\nInventory accumulation may signal slower demand, subsequent markdowns or working-capital pressure. Expansion, acquisitions, deliberate replenishment and product mix can produce similar signals with different outcomes. A positive gap is a research flag whose interpretation requires context.",
    "Sample and Review Coverage": f"The {company_count}-company candidate universe was manually assembled across five researcher-defined business groups: apparel/footwear/off-price, department stores, home/furniture/home improvement, sporting/electronics/other merchandise, and beauty/personal care. It is not a census, random sample or certified GICS classification. U.S. listing does not imply exclusively U.S. operations or pure retail revenue.\n\nCompany identity is mapped to SEC CIK. Company Facts JSON snapshots supply standard USD facts. Annual revenue periods in 10-K/10-K/A filings anchor the panel; the extraction matches periods and accession numbers, distinguishing flow measures from year-end balances. The earliest visible filing in the cached facts is an extraction rule, not historical point-in-time certification.\n\nOutcome-specific usable annual coverage of at least five years, continuity, numeric checks and documented scope restrictions define an unbalanced panel. The pipeline preserves source tags, limited flagged derivations, original amounts and separate analysis values. Module 9 reconciled identified reporting differences, including DDS net-sales denominators and separately presented gross-profit charges. Other scope and classification differences remain subject to review.\n\nReviewed pairs denote recorded company eligibility plus prior/current/next annual-scope checks. This is a research review status, not an independent audit. No new metric values were assumed for pending decisions. Applied metric values: {len(tables['applied'])}.",
    "Methods": "The project compares positive and nonpositive inventory gaps, examines correlations and company-level patterns, and estimates OLS models on company-year pairs. Company-balanced descriptive comparisons require at least three observations in each gap group for a company.\n\nThe main adjusted model explains the next-year ratio using inventory-growth gap, the current ratio, revenue growth, log total assets, company fixed effects and target-end-calendar-year fixed effects. Inference uses firm-clustered CRV1 covariance with small-sample corrections and t-based 95% intervals. The estimates are scaled to a 10 percentage-point inventory gap.\n\nCurrent-ratio controls address starting-level differences and mean-reversion patterns. Baseline-controlled change and next-level specifications have an algebraic coefficient relationship; they are not independent replications. Company/year controls reduce some confounding but do not identify causal effects. Calendar-year effects do not fully align fiscal operating periods.\n\nSensitivity rules examine all pairs, periods without length changes exceeding three days, common outcome pairs, and central 98% of outcome-specific inventory gaps. Trimming uses the signal, not future outcomes. R-squared measures in-sample fit and is not predictive accuracy.",
    "Descriptive Results": "The tables distinguish pooled annual-pair summaries from company-equal comparisons. Mean changes, median changes and decline frequencies capture different distribution features. A positive group difference means a more positive next-year change in Gap > 0 relative to Gap <= 0.\n\nThese descriptive differences are not causal effects and do not incorporate the full control set. A descriptive positive cash-flow association can coexist with an imprecise adjusted estimate; the full model comparison should be consulted.",
    "Adjusted Regression Results": "\n\n".join(adjusted_sentences) + "\n\nThe fully adjusted estimates should be interpreted alongside the alternative control specifications in the figure. Any conclusion is conditional on the provisional sample, measured variables and inference assumptions.",
    "Sensitivity Analysis": " ".join(sensitivity_notes) + "\n\n" + final_conclusion + " Smaller subsets also change sample composition and precision. The four rules are sensitivity checks, not independent experiments.",
    "Filing-reviewed Cases": "Cases were selected from available research pairs using subgroup maximum current gaps, a representative near the median positive gap, and the minimum gap. Selection did not use subsequent outcome direction, although entry into the research-pair pool depends on outcome-data availability. These purposefully selected cases cannot estimate population frequencies.\n\nThe table and evidence narratives include only cases marked reviewed. Management explanations, statement facts and research interpretation should be distinguished. Retain means no additional exclusion was justified by the recorded review; it does not certify causality.",
    "Research Limits and Remaining Work": f"The manually selected present-day candidate list can create selection and survivorship bias. Unbalanced history and outcome-dependent missingness affect generalization. Cost and gross-profit classifications differ across firms; fixed effects do not remove every accounting-comparability problem. Missing annual cost fields: {missing_cost}; missing annual gross-profit fields: {missing_gp}.\n\nFiling review is partial. Pending selected cases: {pending_names}. Most annual rows and company histories have not completed scope review. Recorded derivations and standard-tag selections do not constitute complete historical verification.\n\nCurrent filings arrive {timing_min}-{timing_max} days after the target year begins in these retained samples. The design is retrospective and may overlap the target operating period. It is not a before-year-start forecast. Latest cached comparatives and incomplete historical information sets prevent point-in-time certification.\n\nThe project has no out-of-sample prediction validation, stock-return backtest or causal identification. Small business groups and a limited number of firm clusters constrain inference. Wide intervals can reflect limited precision; they do not prove absence of an effect.\n\nFurther work can complete pending case reviews, prioritize material missing fields, assess same-store sales and store expansion, and define an explicit filing-date-based forecast horizon before attempting predictive validation. These extensions are not claimed as completed in this version.",
    "Practical Use and Conclusion": "The practical contribution is a reproducible fundamental-research workflow connecting a transparent inventory signal to accounting checks, subsequent operating ratios and cited filing evidence. It can help prioritize companies for deeper investigation of demand, replenishment, expansion, vendor terms and cash-flow composition.\n\n" + final_conclusion + " Inventory-growth gaps should be interpreted with business context and a fuller information set. This version completes the analysis/report deliverable with explicit remaining-review limits; it does not establish a trading rule or validated prediction model.",
    "Sources and Reproduction": f"Primary inputs are cached SEC Company Facts and annual filings. Amounts use USD; growth rates and ratios are stored as decimals and displayed as percent or percentage points as labeled. Applied-review snapshot: {metadata[12]['processed_at_utc']}. Statistical results were regenerated from that snapshot through Modules 10 and 11.\n\nThe companion source_manifest.json identifies every input file with a SHA-256 hash. Reviewed-case links below point to the original filings; detailed accounting and annual-scope evidence remain in the input audit tables. README.md describes execution order and deliverable structure. The report date labels this deliverable, not the underlying financial snapshot.",
}
references = [
    ("SEC: EDGAR Application Programming Interfaces", "https://www.sec.gov/search-filings/edgar-application-programming-interfaces"),
    ("Gaur, Fisher, and Raman (2005). An Econometric Analysis of Inventory Turnover Performance in Retail Services. Management Science, 51(2), 181-194.", "https://pubsonline.informs.org/doi/10.1287/mnsc.1040.0298"),
]
report_sections["Research Context and References"] = "Gaur, Fisher, and Raman (2005) examine retail inventory turnover in relation to gross margin, capital intensity and sales surprise. Their study provides background for considering multiple operating factors when interpreting inventory metrics. The present project evaluates subsequent operating ratios using a different inventory-growth-gap measure.\n\nSEC documentation describes the Company Facts API as aggregating standard-taxonomy facts applying to the entire filing entity. Custom-tag disclosures can therefore require direct filing review. The project records unresolved fields and selected filing evidence."
report_tables = {
    "Sample and Review Coverage": [sample_summary, review_summary],
    "Descriptive Results": [pooled_table.round(4), balanced_table.round(4)],
    "Adjusted Regression Results": [main_table.round(4)],
    "Sensitivity Analysis": [sensitivity_table.round(4)],
    "Filing-reviewed Cases": [case_table.round(4)],
    "Research Limits and Remaining Work": [pending_table],
}
section_images = {"Descriptive Results": "group_means.png", "Adjusted Regression Results": "model_comparison.png", "Sensitivity Analysis": "sensitivity.png"}
report_title = "Inventory Growth and Subsequent Operating Performance"
status_line = f"Final report | Historical associations | Partial filing review | Report date: {REPORT_DATE}"
markdown_parts = [f"# {report_title}\n\n{status_line}\n\nSelected U.S.-listed merchandise retailers and consumer brands.\n\n"]
html_parts = [f"<header><p class='eyebrow'>RETAIL FUNDAMENTALS RESEARCH</p><h1>{escape(report_title)}</h1><p>{escape(status_line)}</p><p>Selected U.S.-listed merchandise retailers and consumer brands.</p></header>"]
for title, text in report_sections.items():
    text = text.replace("\\n", "\n")
    markdown_parts.append(f"## {title}\n\n{text}\n\n")
    html_parts.append(f"<section><h2>{escape(title)}</h2>")
    for paragraph in text.split("\n\n"):
        html_parts.append("<p>" + escape(paragraph) + "</p>")
    for table in report_tables.get(title, []):
        if table.empty:
            continue
        markdown_parts.append("| " + " | ".join(table.columns) + " |\n")
        markdown_parts.append("| " + " | ".join(["---"] * len(table.columns)) + " |\n")
        for _, row in table.fillna("").iterrows():
            markdown_parts.append("| " + " | ".join(str(value).replace("|", "/").replace("\n", " ") for value in row) + " |\n")
        markdown_parts.append("\n")
        html_parts.append("<div class='table-wrap'>" + table.to_html(index=False, border=0, escape=True) + "</div>")
    if title in section_images:
        filename = section_images[title]
        encoded = base64.b64encode((FIGURE_DIR / filename).read_bytes()).decode("ascii")
        markdown_parts.append(f"![{title}](figures/{filename})\n\n")
        html_parts.append(f"<figure><img alt='{escape(title)}' src='data:image/png;base64,{encoded}'><figcaption>{escape(caption)}</figcaption></figure>")
    if title == "Filing-reviewed Cases":
        for _, row in reviewed_cases.iterrows():
            label = f"{row['ticker']} | Current period end {row['period_end']} | Next period end {row['next_period_end']}"
            markdown_parts.append(f"### {label}\n\n")
            html_parts.append(f"<article class='case'><h3>{escape(label)}</h3>")
            for field, label_text in [("units_and_scope_notes", "Units and scope"), ("revenue_and_inventory_drivers", "Current drivers"), ("next_margin_and_ocf_drivers", "Next-year drivers"), ("review_notes", "Review interpretation")]:
                note = str(row[field])
                markdown_parts.append(f"**{label_text}:** {note}\n\n")
                html_parts.append(f"<p><strong>{escape(label_text)}:</strong> {escape(note)}</p>")
            for field, label_text in [("current_evidence_url", "Current annual filing"), ("next_evidence_url", "Next annual filing")]:
                url = str(row[field])
                markdown_parts.append(f"[{label_text}]({url})\n\n")
                html_parts.append(f"<p><a href='{escape(url, quote=True)}'>{escape(label_text)}</a></p>")
            html_parts.append("</article>")
    if title == "Research Context and References":
        for label, url in references:
            markdown_parts.append(f"- [{label}]({url})\n\n")
            html_parts.append(f"<p><a href='{escape(url, quote=True)}'>{escape(label)}</a></p>")
    html_parts.append("</section>")
markdown_report = "".join(markdown_parts)
style = "body{font-family:Arial,sans-serif;color:#22313a;background:#fff;margin:0;line-height:1.65}main{max-width:1080px;margin:auto;padding:40px 28px 70px}header{border-bottom:2px solid #245a73;padding-bottom:24px}h1{font-size:30px;line-height:1.2;max-width:820px}h2{font-size:21px;margin:35px 0 12px}h3{font-size:17px}p{margin:12px 0}.eyebrow{font-size:12px;letter-spacing:1px;color:#245a73}.table-wrap{overflow-x:auto;margin:20px 0}table{width:100%;border-collapse:collapse;font-size:13px}th,td{text-align:left;vertical-align:top;padding:9px 10px;border-bottom:1px solid #d8dfe3}th{background:#eef3f5}img{width:100%;height:auto}figure{margin:24px 0}figcaption{font-size:11px;color:#52616b}.case{border-top:1px solid #d8dfe3;margin-top:25px;padding-top:12px}a{color:#245a73}@media print{main{padding:0}h2,h3{break-after:avoid}figure{break-inside:avoid}.table-wrap{overflow:visible}table{font-size:10px}a{color:inherit}}"
html_report = f"<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>{escape(report_title)}</title><style>{style}</style></head><body><main>{''.join(html_parts)}</main></body></html>"
(REPORT_DIR / "Retail_Inventory_Research_Report.md").write_text(markdown_report, encoding="utf-8")
(REPORT_DIR / "Retail_Inventory_Research_Report.html").write_text(html_report, encoding="utf-8")
source_manifest = {"report_date": REPORT_DATE, "generated_at_utc": datetime.now(timezone.utc).isoformat(), "sample_status": metadata[12]["sample_status"], "manual_review_decisions_applied": metadata[12]["manual_review_decisions_applied"], "accounting_scope_verified": metadata[12]["accounting_scope_verified"], "point_in_time_verified": False, "prediction_validated": False, "causal_identification": False, "references": references, "sources": []}
for path in source_paths:
    source_manifest["sources"].append({"path": str(path.relative_to(PROJECT_ROOT)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
(REPORT_DIR / "source_manifest.json").write_text(json.dumps(source_manifest, indent=2), encoding="utf-8")
print("English Markdown and standalone HTML reports saved.")


## Cell 8: Save Reproduction Notes and Confirm the Deliverables

README 明确依赖与重跑顺序，保留当前审核限制。报告在你的实际项目根目录 outputs/retail_inventory_final，Guide 和完整 notebook 在本聊天工作区 outputs。
检查文件已生成且非空，显示英文摘要和主要结果；不会把“文件保存成功”解释为研究假设成立。
这一步完成本版本交付。后续审核或扩展属于下一版，可继续用同一模块更新报告。


In [ ]:
readme = f"""# Retail Inventory Research: Final Deliverable

Report date: {REPORT_DATE}
Scope: retrospective operating-performance associations with partial filing review.

## Files
- Retail_Inventory_Research_Report.html: standalone report with embedded figures.
- Retail_Inventory_Research_Report.md: editable report; keep the figures directory beside it.
- figures/: model comparison, descriptive group means, and adjusted sensitivity intervals.
- source_manifest.json: relative input paths and SHA-256 hashes for this report version.

## Reproduction
Use the existing project root and cached SEC data. Required libraries are pandas, matplotlib, requests, numpy, statsmodels, and IPython.
The completed notebook preserves the earlier modules and their outputs, and appends the executed report module.
Its established-project execution order is Modules 1-9, 12, 10, 11, 13. Module 12 uses the initial Module 11 case file already present in the project.
For a fresh project, first follow the original Guides through Module 11 to create inputs and the initial case file; then run Module 12, Module 10, Module 11, and Module 13.
Set PROJECT_ROOT for your machine. Update REPORT_DATE when generating a new report version.
After changing manual review CSVs, run Module 12 to apply decisions, Modules 10 and 11 to refresh analyses, and all eight Module 13 cells to rebuild the report.
Module 13 alone reads saved inputs; it does not acquire data, audit filings, or refit models.
The version checks intentionally stop reporting if saved analyses or manual decisions are stale.

## Completed and Remaining Work
The analysis/report deliverable is complete for this snapshot. Full historical filing review, prediction validation, and causal identification are not complete.
Reviewed cases: {len(reviewed_cases)} of {len(tables['cases'])}. Pending selected cases: {pending_names}.
The report records pending work and does not fabricate supporting evidence or fill unapproved values.
"""
(REPORT_DIR / "README.md").write_text(readme, encoding="utf-8")
deliverables = ["Retail_Inventory_Research_Report.md", "Retail_Inventory_Research_Report.html", "source_manifest.json", "README.md", "figures/model_comparison.png", "figures/group_means.png", "figures/sensitivity.png"]
for filename in deliverables:
    path = REPORT_DIR / filename
    assert path.exists() and path.stat().st_size > 0, "Missing deliverable: " + filename
    print("Saved:", path)
display(Markdown("## Executive Summary\n\n" + report_sections["Executive Summary"].replace("\\n", "\n")))
display(main_table.round(4))
print("Report version complete; remaining review limits are disclosed.")


## How to Use This Version

1. 可直接先打开已生成的英文 HTML 报告查看结果；它不需要再次下载数据。
2. 把本 Guide 的 8 个 Code Cells 放到现有 notebook 的 Module13 下，按顺序运行。也可使用提供的完整 notebook。
3. 给导师讲时，先说研究问题，再说样本和信息时点，然后展示控制后的区间与 SPWH／URBN 案例，最后说明实际用途及限制。
4. 请用“最终报告已完成，统计样本仍为暂定且年报审核部分完成”，不要说“42 家全历史均已审计”或“证明库存没有用”。
5. 三张图、主要数字、已核查案例和来源清单已经形成可交付版本；后续无需为了项目有结论而寻找显著关系。
